# 02 — Feature Engineering & Leakage Controls

EOR Atlas research-layer notebook. Run this notebook from VS Code/Jupyter.


In [1]:

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
import hashlib
import json
import re

import numpy as np
import pandas as pd


SEED = 42

# Locate the repository root robustly from a notebook working directory.
def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "src").exists() and (p / "outputs").exists():
            return p
    # Fallback for running the notebook from src/notebooks.
    return Path.cwd().parents[2]


PROJECT_ROOT = find_project_root()
ML_DATA_DIR = PROJECT_ROOT / "src" / "notebooks" / "ml_data"
ARTIFACT_DIR = PROJECT_ROOT / "outputs" / "model_artifacts"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "ml_research"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


COLUMN_ALIASES = {
    "technique": [
        "EOR technique", "EOR Technique", "technique",
        "Technique", "EOR_Technique", "EOR Method", "Method"
    ],
    "formation": [
        "Formation type", "formation_category", "Formation",
        "Formation Type", "Rock Type", "rock_type"
    ],
    "depth_min": ["Depth min (ft)", "depth_min_ft", "Depth_min_ft", "depth_min"],
    "depth_max": ["Depth max (ft)", "depth_max_ft", "Depth_max_ft", "depth_max"],
    "por_min": ["Porosity min (%)", "porosity_min_pct", "por_min"],
    "por_max": ["Porosity max (%)", "porosity_max_pct", "por_max"],
    "perm_min": ["Permeability min (mD)", "perm_min_md", "permeability_min_md", "perm_min"],
    "perm_max": ["Permeability max (mD)", "perm_max_md", "permeability_max_md", "perm_max"],
    "api_min": ["Oil gravity min (°API)", "api_min", "API min", "oil_api_min"],
    "api_max": ["Oil gravity max (°API)", "api_max", "API max", "oil_api_max"],
    "visc_min": ["Oil viscosity min (cp)", "visc_min_cp", "viscosity_min_cp", "visc_min"],
    "visc_max": ["Oil viscosity max (cp)", "visc_max_cp", "viscosity_max_cp", "visc_max"],
    "so_min": ["So at start min (%)", "so_start_min_pct", "so_min"],
    "so_max": ["So at start max (%)", "so_start_max_pct", "so_max"],
}


def normalize_col(s: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(s).strip().lower()).strip("_")


def find_column(df: pd.DataFrame, aliases: List[str]) -> Optional[str]:
    exact = {str(c).strip().lower(): c for c in df.columns}
    for a in aliases:
        if a.strip().lower() in exact:
            return exact[a.strip().lower()]
    normalized = {normalize_col(c): c for c in df.columns}
    for a in aliases:
        n = normalize_col(a)
        if n in normalized:
            return normalized[n]
    return None


def discover_candidate_tables() -> List[Dict[str, Any]]:
    candidates = []
    if not ML_DATA_DIR.exists():
        raise FileNotFoundError(f"ML data directory not found: {ML_DATA_DIR}")

    for path in sorted(ML_DATA_DIR.glob("*.xlsx")):
        try:
            xls = pd.ExcelFile(path)
        except Exception as exc:
            candidates.append({"path": path, "error": str(exc)})
            continue

        for sheet in xls.sheet_names:
            try:
                df = pd.read_excel(path, sheet_name=sheet)
            except Exception as exc:
                candidates.append({
                    "path": path, "sheet": sheet, "error": str(exc)
                })
                continue

            target = find_column(df, COLUMN_ALIASES["technique"])
            formation = find_column(df, COLUMN_ALIASES["formation"])
            range_hits = sum(
                find_column(df, COLUMN_ALIASES[key]) is not None
                for key in COLUMN_ALIASES
                if key not in {"technique", "formation"}
            )

            candidates.append({
                "path": path,
                "sheet": sheet,
                "rows": len(df),
                "columns": len(df.columns),
                "target_column": target,
                "formation_column": formation,
                "range_column_hits": range_hits,
            })

    return candidates


def select_training_table(preferred_name: str = "Dataset_01_EOR.xlsx") -> Tuple[pd.DataFrame, Dict[str, Any]]:
    candidates = discover_candidate_tables()

    valid = [
        c for c in candidates
        if c.get("target_column") is not None
        and c.get("formation_column") is not None
        and c.get("range_column_hits", 0) >= 10
        and "error" not in c
    ]
    if not valid:
        raise ValueError(
            "No training table with an EOR technique target and the expected "
            "reservoir range columns was found in ml_data/."
        )

    preferred = [
        c for c in valid
        if Path(c["path"]).name.lower() == preferred_name.lower()
    ]
    chosen = preferred[0] if preferred else max(valid, key=lambda x: x["range_column_hits"])

    df = pd.read_excel(chosen["path"], sheet_name=chosen["sheet"])
    return df, chosen


def standardize_training_table(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)

    for target_name, aliases in COLUMN_ALIASES.items():
        source = find_column(df, aliases)
        if source is not None:
            out[target_name] = df[source]

    required = [
        "technique", "formation",
        "depth_min", "depth_max",
        "por_min", "por_max",
        "perm_min", "perm_max",
        "api_min", "api_max",
        "visc_min", "visc_max",
        "so_min", "so_max",
    ]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Training table is missing required fields: {missing}")

    for c in required:
        if c not in {"technique", "formation"}:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    out["technique"] = out["technique"].astype(str).str.strip()
    out["formation"] = (
        out["formation"].astype(str).str.strip()
        .replace({
            "Carbonate": "Carbonates",
            "Carbonates": "Carbonates",
            "Unconsolidated Sand": "Unconsolidated sands",
            "Unconsolidated sands": "Unconsolidated sands",
            "Sandstone": "Sandstone",
        })
    )

    out = out.replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=required).copy()

    # Remove clearly invalid range records.
    pairs = [
        ("depth_min", "depth_max"),
        ("por_min", "por_max"),
        ("perm_min", "perm_max"),
        ("api_min", "api_max"),
        ("visc_min", "visc_max"),
        ("so_min", "so_max"),
    ]
    for lo, hi in pairs:
        out = out[out[hi] >= out[lo]]

    # Stable record identifier for duplicate checks.
    out["record_fingerprint"] = [
        hashlib.sha1(
            "|".join(str(v) for v in row).encode("utf-8")
        ).hexdigest()[:16]
        for row in out[required].itertuples(index=False, name=None)
    ]

    return out.reset_index(drop=True)


def midpoint(min_series: pd.Series, max_series: pd.Series) -> pd.Series:
    return (min_series + max_series) / 2.0


def span(min_series: pd.Series, max_series: pd.Series) -> pd.Series:
    return (max_series - min_series).clip(lower=0.0)


def build_feature_table(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series, List[str]]:
    """
    Primary ML representation.

    We deliberately do NOT include:
      - '# projects' because it encodes class frequency rather than reservoir state.
      - raw_line/text identifiers.
      - fuzzy scores in the primary classifier because those may be derived from
        the same literature corpus and can create target leakage.

    Features:
      6 midpoints + 6 spans + log10(permeability) + log10(viscosity)
      + 3 formation one-hot features.
    """
    X = pd.DataFrame(index=df.index)

    X["depth_mid_ft"] = midpoint(df["depth_min"], df["depth_max"])
    X["depth_span_ft"] = span(df["depth_min"], df["depth_max"])

    X["porosity_mid_pct"] = midpoint(df["por_min"], df["por_max"])
    X["porosity_span_pct"] = span(df["por_min"], df["por_max"])

    X["perm_mid_md"] = midpoint(df["perm_min"], df["perm_max"])
    X["perm_span_md"] = span(df["perm_min"], df["perm_max"])

    X["api_mid"] = midpoint(df["api_min"], df["api_max"])
    X["api_span"] = span(df["api_min"], df["api_max"])

    X["visc_mid_cp"] = midpoint(df["visc_min"], df["visc_max"])
    X["visc_span_cp"] = span(df["visc_min"], df["visc_max"])

    X["so_mid_pct"] = midpoint(df["so_min"], df["so_max"])
    X["so_span_pct"] = span(df["so_min"], df["so_max"])

    X["log10_perm_mid"] = np.log10(np.clip(X["perm_mid_md"], 1e-6, None))
    X["log10_visc_mid"] = np.log10(np.clip(X["visc_mid_cp"], 1e-6, None))

    for formation in ["Sandstone", "Carbonates", "Unconsolidated sands"]:
        X[f"formation_{normalize_col(formation)}"] = (
            df["formation"].eq(formation).astype(float)
        )

    X = X.astype(float)
    y = df["technique"].copy()
    return X, y, list(X.columns)


def top_k_accuracy(y_true: np.ndarray, proba: np.ndarray, k: int = 3) -> float:
    pred = np.argsort(proba, axis=1)[:, -k:]
    y_true = np.asarray(y_true)
    return float(np.mean([y in row for y, row in zip(y_true, pred)]))


## 1. Load and standardize

In [2]:
raw_df, chosen = select_training_table()
df = standardize_training_table(raw_df)
print(chosen)
print(df.shape)


{'path': WindowsPath('c:/Users/mnabielizzuddin.radz/OneDrive - PETRONAS/Reservoir Engineering/Programming_Python_Projects/EOR ATLAS/EORWEB/EORWEBDEV/src/notebooks/ml_data/NeuroFuzzy_EOR_Extracted_Tables.xlsx'), 'sheet': 'Table1_Ranges', 'rows': 24, 'columns': 17, 'target_column': 'EOR technique', 'formation_column': 'Formation type', 'range_column_hits': 12}
(17, 15)


## 2. Build the primary non-fuzzy feature space

In [3]:
X, y_text, FEATURE_NAMES = build_feature_table(df)
print('Feature matrix:', X.shape)
print('Target classes:', y_text.nunique())
display(X.head())
display(y_text.value_counts().rename('count').to_frame())


Feature matrix: (17, 17)
Target classes: 10


,depth_mid_ft,depth_span_ft,porosity_mid_pct,porosity_span_pct,perm_mid_md,perm_span_md,api_mid,api_span,visc_mid_cp,visc_span_cp,so_mid_pct,so_span_pct,log10_perm_mid,log10_visc_mid,formation_sandstone,formation_carbonates,formation_unconsolidated_sands
0,3000.0,5500.0,27.0,24.0,5050.00,9900.0,15.0,14.0,250009.00,499982.00,55.0,70.0,3.703291,5.397956,1.0,0.0,0.0
1,1662.5,2975.0,32.5,15.0,7650.00,14700.0,17.0,16.0,100087.50,199825.00,69.0,42.0,3.883661,5.000380,0.0,0.0,1.0
2,1025.0,950.0,42.5,45.0,1000.50,1999.0,19.5,19.0,2013.00,3974.00,65.0,40.0,3.000217,3.303844,0.0,1.0,0.0
3,6775.0,10350.0,19.0,18.0,1154.50,2291.0,36.0,18.0,1.65,2.70,51.5,51.0,3.062394,0.217484,1.0,0.0,0.0
4,7550.0,7100.0,14.0,20.0,2500.05,4999.9,36.5,17.0,3.16,5.68,59.5,59.0,3.397949,0.499687,0.0,1.0,0.0


,count
technique,
Steam,3
Combustion,3
Miscible CO2,2
Miscible HC,2
Surfactants,2
Polymer,1
Nitrates,1
Microbial,1
Hot water,1


In [4]:
X.to_csv(PROCESSED_DIR / 'X_primary.csv', index=False)
y_text.to_csv(PROCESSED_DIR / 'y_text.csv', index=False, header=['technique'])
df.to_csv(PROCESSED_DIR / 'standardized_records.csv', index=False)
json.dump(FEATURE_NAMES, open(PROCESSED_DIR / 'feature_names.json', 'w', encoding='utf-8'), indent=2)
print('Saved processed research data to:', PROCESSED_DIR)


Saved processed research data to: c:\Users\mnabielizzuddin.radz\OneDrive - PETRONAS\Reservoir Engineering\Programming_Python_Projects\EOR ATLAS\EORWEB\EORWEBDEV\outputs\ml_research


## Important leakage decision
The primary benchmark does not feed fuzzy scores into the classifier. The fuzzy layer is evaluated separately in EOR Intelligence. This prevents the classifier from simply learning the same literature-derived envelope logic that generated the training labels.